# Specular-Gaussians — Mip-NeRF 360 Multi-Resolution Sweep & Upload Pipeline

> **Kernel**: `thesis_env` (Set via Kernel → Change Kernel after running `bosch_setup_thesis.ipynb` once to register it)

Runs the full Specular-Gaussians baseline training sweep on Mip-NeRF 360 scenes inside the BOSCH server environment across multiple selected image resolutions (`images`, `images_2`, `images_4`, `images_8`), automatically archiving, uploading results to Hugging Face, and purging local output files to save disk space.

**What this notebook does:**
1. **Proxy & Env Check**: Sets up environment variables for the BOSCH server.
2. **Target Resolutions Selection**: Choose any combination of resolutions (`["images", "images_2", "images_4", "images_8"]`).
3. **Resolution & Layout Validation**: Verifies selected resolution folders exist for all 9 scenes (`bicycle`, `flowers`, `garden`, `stump`, `treehill`, `room`, `counter`, `kitchen`, `bonsai`).
4. **Sweep Execution**: Invokes `run_mip360.sh` in `Specular-Gaussians` for each selected resolution with `DATA_ROOT="/home/ghp4hc/datasets/datasets/mipneft360"`.
5. **Results Summary**: Formats and prints quantitative metrics (`results.json`) in a neat table for each resolution.
6. **Archiving & HF Upload**: Zips and uploads each resolution's output (`images.zip`, `images2.zip`, `images4.zip`, `images8.zip`) directly to `DiBiay/specular_gaussians-mipnerf360-result`.
7. **Auto Cleanup**: Purges the local zip archive and output directory after a successful upload to save BOSCH server disk space.

## c00 — Proxy Settings
Sets the BOSCH proxy for external connectivity.

In [1]:
# ── Proxy (required for HF / huggingface cache / diagnostic endpoints) ────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

Proxy set to: http://rb-proxy-sl.bosch.com:8080


## c01 — Config, Target Resolutions & Kernel Check
Defines paths, target resolution list (`["images", "images_2", "images_4", "images_8"]`), and double-checks if the correct virtual environment kernel is loaded.

In [ ]:
# ── Configurations & environment variables check ─────────────────────────────
import os
import sys

HOME = os.path.expanduser('~')

# Only running images_2 resolution
TARGET_RESOLUTIONS = ["images_2"]

# Robustly resolve Specular-Gaussians repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/Specular-Gaussians_backup_v2'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/Specular-Gaussians_backup_v2'
elif os.path.isdir('/home/ghp4hc/thesis-all/Specular-Gaussians'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/Specular-Gaussians'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians_backup_v2')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians_backup_v2')
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'Specular-Gaussians')

ENV_NAME = 'thesis_env'

print(f'Active Python        : {sys.executable}')
print(f'Active Kernel name   : {ENV_NAME}')
print(f'Repository Root      : {REPO_ROOT}')
print(f'Target Resolutions   : {TARGET_RESOLUTIONS}')

assert REPO_ROOT in sys.executable or ENV_NAME in sys.executable or '.conda' in sys.executable, \
    f"WARNING: You are not running on the '{ENV_NAME}' kernel! Please select Kernel -> Change Kernel -> Python ({ENV_NAME})"

## c02 — Imports & GPU Validation
Verifies hardware detection and compiled custom modules availability.

In [ ]:
# ── Verification of PyTorch & custom submodules ────────────────────────────────
import subprocess
import sys
import os
import glob
import torch

def ensure_importable(pkg, import_name=None, extra_pip_args=None):
    """Import a package, reinstalling it if the import fails (missing or broken binary)."""
    import_name = import_name or pkg
    try:
        __import__(import_name)
        return True
    except Exception as e:
        print(f'{import_name}: reinstalling ({e.__class__.__name__}: {e})')
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg], capture_output=True)
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--proxy', os.environ.get('HTTPS_PROXY', '')]
                           + (extra_pip_args or []), capture_output=True, text=True)
        print(r.stdout[-500:] if r.returncode == 0 else r.stderr[-500:])
        return r.returncode == 0

# Same nvcc search as bosch_setup_thesis.ipynb's c03_cuda_search (proven to work on this server):
# tries module-load first, then falls back to scanning common HPC mount points.
CUDA_SEARCH_SCRIPT = r'''
which nvcc 2>/dev/null && exit 0
for init in /etc/profile /etc/profile.d/modules.sh \
            /usr/share/lmod/lmod/init/bash /usr/share/lmod/lmod/init/sh; do
    [ -f "$init" ] && source "$init" 2>/dev/null
done
for mod in cuda/11.7 cuda/11.8 cuda/11.2 cuda/11 cuda CUDA/11.7 CUDA/11.8 CUDA cuda-11.7 cuda-11.8 cuda-11 cuda/12.6 cuda/12.1 cuda/12 cuda CUDA/12.6 CUDA/12.1 cuda-12.6 cuda-12-1 cuda-12; do
    module load "$mod" 2>/dev/null
    nv=$(which nvcc 2>/dev/null); [ -n "$nv" ] && echo "$nv" && exit 0
done
for base in /fs /work /gpfs /scratch /software /apps /appl /tools /opt/software /usr/local; do
    [ -d "$base" ] || continue
    result=$(find "$base" -name nvcc -type f -maxdepth 8 2>/dev/null | head -1)
    [ -n "$result" ] && echo "$result" && exit 0
done
'''

def find_cuda_home():
    cuda_home = os.environ.get('CUDA_HOME', '')
    if cuda_home and os.path.isfile(os.path.join(cuda_home, 'bin', 'nvcc')):
        return cuda_home
    r_nvcc = subprocess.run(['bash', '-c', CUDA_SEARCH_SCRIPT], capture_output=True, text=True, timeout=90)
    nvcc_path = r_nvcc.stdout.strip()
    if not nvcc_path or not os.path.isfile(nvcc_path):
        raise RuntimeError('nvcc not found — check CUDA module availability on server')
    return os.path.dirname(os.path.dirname(nvcc_path))

def build_and_install_rasterizer():
    """Build the anchor-based diff_gaussian_rasterization fork (with visible_filter) from source."""
    cuda_home = find_cuda_home()
    cc = f'{torch.cuda.get_device_capability(0)[0]}.{torch.cuda.get_device_capability(0)[1]}' if torch.cuda.is_available() else '8.0'
    src_dir = os.path.join(REPO_ROOT, 'submodules', 'depth-diff-gaussian-rasterization')
    print(f'Building diff_gaussian_rasterization from {src_dir} (CUDA_HOME={cuda_home}, arch={cc}) ...')
    build_cmd = (
        f'cd "{src_dir}" && rm -rf build dist *.egg-info && '
        f'CUDA_HOME={cuda_home} PATH={cuda_home}/bin:$PATH TORCH_CUDA_ARCH_LIST={cc} '
        f'{sys.executable} setup.py bdist_wheel 2>&1'
    )
    r = subprocess.run(build_cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('BUILD FAILED:', r.stdout[-1500:])
        return False
    wheels = glob.glob(f'{src_dir}/dist/*.whl')
    if not wheels:
        print('No wheel produced')
        return False
    ri = subprocess.run([sys.executable, '-m', 'pip', 'install', wheels[0], '--no-deps', '--force-reinstall'],
                        capture_output=True, text=True)
    print(ri.stdout[-500:] if ri.returncode == 0 else ri.stderr[-500:])
    return ri.returncode == 0

print('PyTorch version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name :', torch.cuda.get_device_name(0))
    print('Compute Cap.    :', torch.cuda.get_device_capability(0))

# lpips is required by Specular-Gaussians metrics but not always present in thesis_env
ensure_importable('lpips')

# torch_scatter ships a compiled extension pinned to a specific CUDA build; if the
# installed wheel doesn't match this env's torch+CUDA (e.g. "libcudart.so.12" missing),
# reinstall the wheel matching the *actual* installed torch version.
ensure_importable('torch_scatter', extra_pip_args=['-f', f'https://data.pyg.org/whl/torch-{torch.__version__}.html'])

# This codebase needs the anchor-based rasterizer fork (adds GaussianRasterizer.visible_filter,
# used by prefilter_voxel). A generic/vanilla diff_gaussian_rasterization build lacks that method.
try:
    import diff_gaussian_rasterization as _dgr
    if not hasattr(_dgr.GaussianRasterizer, 'visible_filter'):
        raise AttributeError("installed diff_gaussian_rasterization is missing 'visible_filter' (wrong fork/build)")
except Exception as e:
    print(f'diff_gaussian_rasterization: rebuilding ({e})')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'diff_gaussian_rasterization'], capture_output=True)
    build_and_install_rasterizer()

import diff_gaussian_rasterization
import simple_knn
import lpips
import torch_scatter
assert hasattr(diff_gaussian_rasterization.GaussianRasterizer, 'visible_filter'), \
    'diff_gaussian_rasterization still missing visible_filter after rebuild'
print('rasterizer      : OK (visible_filter present)')
print('simple-knn      : OK')
print('lpips           : OK')
print('torch_scatter   : OK')

try:
    import huggingface_hub
    print('huggingface_hub : OK')
except ImportError:
    print('huggingface_hub : MISSING (will auto-install during the upload step)')

## c03 — Verify Dataset Path
Checks that the Mip-NeRF 360 source dataset is available on the server.

In [4]:
# ── Verify Dataset Directory ──────────────────────────────────────────────────
import os
import sys

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
assert os.path.exists(src_root), f"Dataset path not found at {src_root}! Check that the datasets are downloaded."
print(f"✅ Found dataset source root: {src_root}")

print(f"\n📂 Source datasets directory content:")
print(os.listdir(src_root))

✅ Found dataset source root: /home/ghp4hc/datasets/datasets/mipneft360

📂 Source datasets directory content:
['360_v2', '360_extra_scenes']


## c04 — Verify Target Resolutions Layout
Validates all 9 scenes for each resolution in `TARGET_RESOLUTIONS`.

In [5]:
# ── Verify Mip-NeRF 360 Dataset Layout for all Target Resolutions ────────────
import os

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

for resolution in TARGET_RESOLUTIONS:
    print(f"\n🔍 Verifying resolution '{resolution}' under {src_root} ...")
    missing = []
    for scene in MIP360_SCENES:
        v2_path = os.path.join(src_root, "360_v2", scene)
        extra_path = os.path.join(src_root, "360_extra_scenes", scene)
        
        if os.path.isdir(v2_path):
            scene_dir = v2_path
        elif os.path.isdir(extra_path):
            scene_dir = extra_path
        else:
            scene_dir = None
            
        if scene_dir is None:
            status = "MISSING (scene folder not found)"
            missing.append(scene)
        else:
            images_dir = os.path.join(scene_dir, resolution)
            if not os.path.isdir(images_dir):
                status = f"MISSING ({resolution} not found; has: {sorted(os.listdir(scene_dir))[:6]})"
                missing.append(scene)
            else:
                n_imgs = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                status = f"OK ({n_imgs} images)"
                
        print(f"  {scene:<12s} {status}")

    if missing:
        print(f"⚠️  {len(missing)}/{len(MIP360_SCENES)} scene(s) missing {resolution}: {missing}")
    else:
        print(f"✅ All {len(MIP360_SCENES)} scenes verified for '{resolution}'.")


🔍 Verifying resolution 'images_2' under /home/ghp4hc/datasets/datasets/mipneft360 ...
  bicycle      OK (194 images)
  flowers      OK (173 images)
  garden       OK (185 images)
  stump        OK (125 images)
  treehill     OK (141 images)
  room         OK (311 images)
  counter      OK (240 images)
  kitchen      OK (279 images)
  bonsai       OK (292 images)
✅ All 9 scenes verified for 'images_2'.


## c05 — Run Specular-Gaussians Sweeps, Upload & Auto-Cleanup
Iterates through each resolution in `TARGET_RESOLUTIONS`, runs `run_mip360.sh`, summarizes metrics, zips output, uploads to Hugging Face, and purges local zip & output folder to save disk space.

In [ ]:
# ── Prepare script inputs & HF API ───────────────────────────────────────────
import subprocess
import os
import sys
import json
import shutil

# Ensure BOSCH Proxy environment variables & explicit proxy dictionary are set
PROXY = 'http://rb-proxy-sl.bosch.com:8080'
os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

PROXIES_DICT = {
    'http': PROXY,
    'https': PROXY,
}

try:
    import huggingface_hub
    from huggingface_hub import HfApi
except ImportError:
    print("Installing huggingface_hub via pip...")
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "--proxy", PROXY], check=True)
    import huggingface_hub
    from huggingface_hub import HfApi

# Configure global HTTP backend for huggingface_hub with BOSCH proxy
def proxy_session_factory():
    import requests
    session = requests.Session()
    session.proxies = PROXIES_DICT
    return session

try:
    huggingface_hub.configure_http_backend(backend_factory=proxy_session_factory)
except Exception:
    pass

# Robustly resolve Specular-Gaussians repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/Specular-Gaussians_backup_v2'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/Specular-Gaussians_backup_v2'
elif os.path.isdir('/home/ghp4hc/thesis-all/Specular-Gaussians'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/Specular-Gaussians'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians_backup_v2')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians_backup_v2')
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'Specular-Gaussians')

DATA_ROOT = "/home/ghp4hc/datasets/datasets/mipneft360"
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
HF_REPO = os.environ.get('HF_REPO', 'DiBiay/spec-gaussian-mipneft360-images2')

venv_bin = os.path.dirname(sys.executable)
cuda_home = os.environ.get('CUDA_HOME', '')
if not cuda_home or not os.path.isfile(os.path.join(cuda_home, 'bin', 'nvcc')):
    # Same nvcc search as bosch_setup_thesis.ipynb's c03_cuda_search (proven to work on this server):
    # tries module-load first, then falls back to scanning common HPC mount points.
    search_script = r'''
    which nvcc 2>/dev/null && exit 0
    for init in /etc/profile /etc/profile.d/modules.sh \
                /usr/share/lmod/lmod/init/bash /usr/share/lmod/lmod/init/sh; do
        [ -f "$init" ] && source "$init" 2>/dev/null
    done
    for mod in cuda/11.7 cuda/11.8 cuda/11.2 cuda/11 cuda CUDA/11.7 CUDA/11.8 CUDA cuda-11.7 cuda-11.8 cuda-11 cuda/12.6 cuda/12.1 cuda/12 cuda CUDA/12.6 CUDA/12.1 cuda-12.6 cuda-12-1 cuda-12; do
        module load "$mod" 2>/dev/null
        nv=$(which nvcc 2>/dev/null); [ -n "$nv" ] && echo "$nv" && exit 0
    done
    for base in /fs /work /gpfs /scratch /software /apps /appl /tools /opt/software /usr/local; do
        [ -d "$base" ] || continue
        result=$(find "$base" -name nvcc -type f -maxdepth 8 2>/dev/null | head -1)
        [ -n "$result" ] && echo "$result" && exit 0
    done
    '''
    r_nvcc = subprocess.run(['bash', '-c', search_script], capture_output=True, text=True, timeout=90)
    if r_nvcc.returncode == 0 and r_nvcc.stdout.strip():
        cuda_home = os.path.dirname(os.path.dirname(r_nvcc.stdout.strip()))
    else:
        raise RuntimeError('nvcc not found — check CUDA module availability on server')

MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

def fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "-"

def count_gaussians(scene_dir):
    ply_path = os.path.join(scene_dir, "point_cloud", "iteration_30000", "point_cloud.ply")
    if os.path.exists(ply_path):
        try:
            with open(ply_path, 'rb') as f:
                for line in f:
                    line_str = line.decode('ascii', errors='ignore')
                    if line_str.startswith("element vertex"):
                        return int(line_str.split()[2])
        except Exception:
            pass
    return "-"

for resolution in TARGET_RESOLUTIONS:
    print(f"\n========================================================================")
    print(f" 🚀 STARTING SPECULAR-GAUSSIANS SWEEP FOR RESOLUTION: {resolution}")
    print(f"========================================================================")
    
    logfile = os.path.join(os.path.dirname(REPO_ROOT), f"mip360_{resolution}_specular_gaussians_run.log")
    
    # 1. Run bash sweep
    cmd = f'''
    export PATH={venv_bin}:{cuda_home}/bin:$PATH
    export LD_LIBRARY_PATH={cuda_home}/lib64:$LD_LIBRARY_PATH
    export CUDA_VISIBLE_DEVICES=0
    export DATA_ROOT={DATA_ROOT}
    export IMAGES={resolution}
    cd "{REPO_ROOT}"
    bash run_mip360.sh > "{logfile}" 2>&1
    '''
    r = subprocess.run(['bash', '-c', cmd])
    
    print(f"--- tail of {logfile} ---")
    if os.path.exists(logfile):
        with open(logfile, 'r') as f:
            lines = f.readlines()
            print(''.join(lines[-40:]))
            
    # 2. Print quantitative summary
    out_root = os.path.join(REPO_ROOT, "output", f"mip360_{resolution}")
    print(f"\n📊 Quantitative Results Summary for '{resolution}':")
    header = f"{'scene':<12s}{'PSNR':>10s}{'SSIM':>10s}{'LPIPS':>10s}{'#Gaussians':>14s}"
    print(header)
    print("-" * len(header))
    psnr_l, ssim_l, lpips_l = [], [], []
    for scene in MIP360_SCENES:
        out_dir = os.path.join(out_root, scene)
        results_path = os.path.join(out_dir, "results.json")
        if not os.path.exists(results_path):
            print(f"{scene:<12s}  (no results.json)")
            continue
        with open(results_path) as f:
            res = json.load(f)
        ik = next(iter(res.keys())) if res else None
        m = res.get(ik, {}) if ik else {}
        pv, sv, lv = m.get("PSNR"), m.get("SSIM"), m.get("LPIPS")
        ng = count_gaussians(out_dir)
        if isinstance(pv, (int, float)): psnr_l.append(pv)
        if isinstance(sv, (int, float)): ssim_l.append(sv)
        if isinstance(lv, (int, float)): lpips_l.append(lv)
        print(f"{scene:<12s}{fmt(pv):>10s}{fmt(sv):>10s}{fmt(lv):>10s}{str(ng):>14s}")
    print("-" * len(header))
    if psnr_l:
        print(f"{'Average':<12s}{fmt(sum(psnr_l)/len(psnr_l)):>10s}{fmt(sum(ssim_l)/len(ssim_l)):>10s}{fmt(sum(lpips_l)/len(lpips_l)):>10s}{'-':>14s}")

    # 3. Push whole output folder to Hugging Face (no zip)
    if os.path.isdir(out_root):
        # Strip point_cloud folders (large, unused after rendering) before uploading
        for scene in MIP360_SCENES:
            pc_dir = os.path.join(out_root, scene, "point_cloud")
            if os.path.isdir(pc_dir):
                shutil.rmtree(pc_dir, ignore_errors=True)

        # Prompt for HF token if not set
        token_to_use = HF_TOKEN
        if not token_to_use:
            token_to_use = input("Enter your Hugging Face Access Token (WRITE permission required): ").strip()

        # 4. Upload to Hugging Face
        if token_to_use:
            print(f"\n📤 Uploading whole folder '{out_root}' -> Hugging Face dataset '{HF_REPO}'...")
            try:
                api = HfApi(proxies=PROXIES_DICT)
                try:
                    api.repo_info(repo_id=HF_REPO, repo_type="dataset", token=token_to_use)
                except Exception as repo_err:
                    if "404" in str(repo_err) or "Repository Not Found" in str(repo_err):
                        print(f"➕ Creating dataset repository '{HF_REPO}'...")
                        api.create_repo(repo_id=HF_REPO, repo_type="dataset", token=token_to_use, private=True)
                    else:
                        print(f"ℹ️ Repo info status note: {repo_err}")

                api.upload_folder(
                    folder_path=out_root,
                    repo_id=HF_REPO,
                    repo_type="dataset",
                    token=token_to_use,
                )
                print(f"🎉 [SUCCESS] Uploaded '{out_root}' to '{HF_REPO}'!")

                # 5. Purge local output directory to prevent OOM & save disk space
                print(f"🗑️ Cleaning up output directory '{out_root}'...")
                shutil.rmtree(out_root, ignore_errors=True)
                print(f"✅ Local disk space freed for resolution '{resolution}'!")
            except Exception as e:
                print(f"❌ [ERROR] HF upload failed for {resolution}: {e}")
                print(f"⚠️  Retaining output folder '{out_root}' for inspection.")
        else:
            print(f"⚠️  Skipping HF upload (HF_TOKEN not set).")
    else:
        print(f"❌ [ERROR] Output directory '{out_root}' not found. Skipping upload for {resolution}.")